In [ ]:
!pip install ptflops

In [ ]:
import torch
import timm
import os
import random
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch.optim as optim
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import GroupKFold
from ptflops import get_model_complexity_info
import time

In [ ]:
# model = timm.create_model('mobilevit_xs', pretrained=True, num_classes=2, features_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# print("Base model loaded successfully!")

In [ ]:
dataset_path = "/kaggle/input/datasets/BreakHis_augmented"

from copy import deepcopy
import torch.nn.functional as F
# Training configuration
num_folds = 10
num_epochs = 30
batch_size = 32
learning_rate = 0.0001
criterion = nn.CrossEntropyLoss()
KD_T = 4.0
KD_ALPHA_LOGIT = 0.3
KD_ALPHA_FEATURE = 0.2
KD_ALPHA_CE = 0.5

criterion = nn.CrossEntropyLoss()
feature_criterion = nn.MSELoss()

transform = transforms.Compose([
    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class MobileViTWithHead(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "mobilevit_xs",
            pretrained=True,
            features_only=True
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(384, 2)

    def forward(self, x):

        feats = self.backbone(x)
        final_feat = feats[-1]

        pooled = self.pool(final_feat).flatten(1)
        logits = self.classifier(pooled)

        return logits, final_feat

model = MobileViTWithHead().to(device)
print("Model loaded")

In [ ]:
def load_images_by_magnification(dataset_path, mag_level):

    images, labels, slide_ids = [], [], []

    for class_name, label in [("benign",0), ("malignant",1)]:

        class_root = os.path.join(dataset_path, class_name, "SOB")

        for subtype in os.listdir(class_root):

            subtype_path = os.path.join(class_root, subtype)

            for slide in os.listdir(subtype_path):

                slide_path = os.path.join(subtype_path, slide)

                mag_folder = os.path.join(slide_path, f"{mag_level}X")

                if not os.path.exists(mag_folder):
                    continue

                for filename in os.listdir(mag_folder):

                    img_path = os.path.join(mag_folder, filename)

                    try:
                        img = Image.open(img_path).convert("RGB").resize((224,224))

                        images.append(transform(img))
                        labels.append(label)
                        slide_ids.append(slide)

                    except Exception as e:
                        print("Error:", e)

    return torch.stack(images), torch.tensor(labels), slide_ids

In [ ]:
images_40x , labels_40x , slides_40x = load_images_by_magnification(dataset_path , 40)

In [ ]:
images_100x , labels_100x , slides_100x = load_images_by_magnification(dataset_path , 100)

In [ ]:
images_200x , labels_200x , slides_200x = load_images_by_magnification(dataset_path , 200)

In [ ]:
images_400x , labels_400x , slides_400x = load_images_by_magnification(dataset_path , 400)

In [ ]:


def train_model(data, model, teacher_model=None):

    images, labels, slide_ids = data

    gkf = GroupKFold(n_splits=num_folds)
    fold_results = []

    initial_weights = deepcopy(model.state_dict())

    if teacher_model:
        teacher_model.eval()
        for p in teacher_model.parameters():
            p.requires_grad = False

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(images, labels, groups=slide_ids)
    ):

        print(f"\n========== Fold {fold+1}/{num_folds} ==========")

        model.load_state_dict(initial_weights)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        x_train, x_test = images[train_idx], images[test_idx]
        y_train, y_test = labels[train_idx], labels[test_idx]

        train_loader = DataLoader(
            TensorDataset(x_train, y_train),
            batch_size=batch_size,
            shuffle=True
        )

        test_loader = DataLoader(
            TensorDataset(x_test, y_test),
            batch_size=batch_size
        )

        # ---------------- TRAIN ----------------
        for epoch in range(num_epochs):

            model.train()

            running_loss = 0
            correct = 0
            total = 0

            for imgs, lbls in train_loader:

                imgs = imgs.to(device).float()
                lbls = lbls.to(device).long()

                optimizer.zero_grad()

                student_logits, student_feat = model(imgs)
                ce_loss = criterion(student_logits, lbls)

                if teacher_model:

                    with torch.no_grad():
                        teacher_logits, teacher_feat = teacher_model(imgs)

                    # LOGIT KD
                    kd_logit = nn.KLDivLoss(reduction="batchmean")(
                        F.log_softmax(student_logits / KD_T, dim=1),
                        F.softmax(teacher_logits / KD_T, dim=1)
                    ) * (KD_T * KD_T)

                    # FEATURE KD
                    kd_feature = feature_criterion(
                        student_feat,
                        teacher_feat
                    )

                    loss = (
                        KD_ALPHA_CE * ce_loss +
                        KD_ALPHA_LOGIT * kd_logit +
                        KD_ALPHA_FEATURE * kd_feature
                    )

                else:
                    loss = ce_loss

                loss.backward()
                optimizer.step()

                _, preds = student_logits.max(1)

                total += lbls.size(0)
                correct += preds.eq(lbls).sum().item()
                running_loss += loss.item()

            print(f"Epoch {epoch+1}/{num_epochs} | "
                  f"Loss {running_loss/len(train_loader):.4f} | "
                  f"Acc {100*correct/total:.2f}%")

        # ---------------- EVAL ----------------
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, lbls in test_loader:

                imgs = imgs.to(device)
                lbls = lbls.to(device)

                logits, _ = model(imgs)
                _, preds = logits.max(1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds)

        fold_results.append({"accuracy":acc, "f1":f1})

        print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    return fold_results

def evaluate_efficiency(model):

    print("\n===== EFFICIENCY =====")

    params = sum(p.numel() for p in model.parameters())
    print(f"Params: {params/1e6:.2f}M")

    macs, _ = get_model_complexity_info(
        model,
        (3,224,224),
        as_strings=True,
        print_per_layer_stat=False
    )

    print("MACs:", macs)

    model_size = sum(p.numel()*4 for p in model.parameters())/(1024**2)
    print(f"Model Size: {model_size:.2f} MB")

    model.eval()
    dummy = torch.randn(1,3,224,224).to(device)

    with torch.no_grad():
        for _ in range(20):
            _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    with torch.no_grad():
        for _ in range(100):
            _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    print(f"Latency: {(end-start)/100*1000:.2f} ms")




print("Training on 40x magnification...")
results_40x = train_model((images_40x , labels_40x , slides_40x) , model)
# Start progressive training using knowledge distillation
print("Training on 100x magnification with knowledge distillation from 40x model...")
model_40x = deepcopy(model)
results_100x = train_model((images_100x, labels_100x, slides_100x), model , teacher_model=model_40x)

print("\nTraining on 200x magnification with knowledge distillation from 100x model...")
model_100x = deepcopy(model)  # Save model state for knowledge distillation
results_200x = train_model((images_200x, labels_200x, slides_200x), model, teacher_model=model_100x)

print("\nTraining on 400x magnification with knowledge distillation from 200x model...")
model_200x = deepcopy(model)  # Update model state for knowledge distillation
results_400x = train_model((images_400x, labels_400x, slides_400x), model, teacher_model=model_200x)


In [ ]:
import torch

# After training finishes
MODEL_PATH = "/kaggle/working/new_KDVIT.pth"
torch.save(model.state_dict(), MODEL_PATH)

print(f"Model saved at {MODEL_PATH}")


In [ ]:
# after final training
evaluate_efficiency(model)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
def evaluate_model_on_preloaded_data(model, data_dict, batch_size=32):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    results = {}

    for mag_level, (images, labels) in data_dict.items():

        print(f"\nEvaluating on {mag_level} magnification...")

        dataset = TensorDataset(images, labels)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        all_preds = []
        all_labels = []

        with torch.no_grad():

            for inputs, targets in dataloader:

                inputs = inputs.to(device)
                targets = targets.to(device)

                # ⭐ FIX HERE
                logits, _ = model(inputs)

                preds = torch.argmax(logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(targets.cpu().numpy())

        accuracy = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="binary")
        precision = precision_score(all_labels, all_preds, average="binary")
        recall = recall_score(all_labels, all_preds, average="binary")

        print(f"Accuracy: {accuracy*100:.2f}%")
        print(f"F1 Score: {f1:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")

        results[mag_level] = {
            "accuracy": accuracy,
            "f1_score": f1,
            "precision": precision,
            "recall": recall,
        }

    return results

data_dict = {
    "40x": (images_40x, labels_40x),
    "100x": (images_100x, labels_100x),
    "200x": (images_200x, labels_200x),
    "400x": (images_400x, labels_400x),
}
# Evaluate the model on preloaded data
results = evaluate_model_on_preloaded_data(model, data_dict)